# 🚕 Urban Flow Analytics — Member 2: Predictive Modeling

**SLIIT Codefest Datathon 2026**

This notebook covers:
- ✅ Task 2.1 — Base Fare Prediction (*No-Surprises Upfront Pricing*)
- ✅ Task 2.2 — Trip Duration Prediction (*On-Time Arrival Estimator*)
- ✅ Model evaluation: RMSE, MAE, R² with metric justification
- ✅ Trained model export in `.pkl` format

## 0. Environment Setup

In [ ]:
# Install required packages (run once)
import subprocess, sys
packages = [
    'pandas', 'numpy', 'scikit-learn', 'xgboost', 'lightgbm',
    'matplotlib', 'seaborn', 'joblib'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages ready!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import xgboost as xgb
import lightgbm as lgb

# Reproducibility
SEED = 42
np.random.seed(SEED)

print(f'pandas {pd.__version__}, sklearn ready, xgb {xgb.__version__}, lgb {lgb.__version__}')

## 1. Data Loading

We load all 12 months of taxi data and the zone lookup table. 
For development we sample 500K rows for speed; set `FULL_DATA = True` for the final run.

In [ ]:
# --- CONFIG ---------------------------------------------------
# CSVs are NOT in GitHub (4.86 GB). Download from organizers.
# Update path in _candidates below if auto-detect fails.
# ---------------------------------------------------------------

FULL_DATA = False
SAMPLE_N  = 500_000
MODEL_DIR = '..\\models'

import pathlib
_candidates = [
    pathlib.Path('..\\..\\Urban_Flow_Analytics_Dataset_csv'),
    pathlib.Path('C:\\data\\Urban_Flow_Analytics_Dataset_csv'),
    pathlib.Path('data\\Urban_Flow_Analytics_Dataset_csv'),
]
_zone_cands = [
    pathlib.Path('..\\..\\Urban_Flow_Analytics_Zone_Dataset.csv'),
    pathlib.Path('C:\\data\\Urban_Flow_Analytics_Zone_Dataset.csv'),
    pathlib.Path('data\\Urban_Flow_Analytics_Zone_Dataset.csv'),
]

DATA_DIR  = next((str(p) for p in _candidates if p.exists()), None)
ZONE_FILE = next((str(p) for p in _zone_cands if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        'CSV data not found! Download from organizers.'
        ' Update _candidates path above.'
    )

os.makedirs(MODEL_DIR, exist_ok=True)
print('DATA_DIR  :', DATA_DIR)
print('ZONE_FILE :', ZONE_FILE)

csv_files = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith('.csv')
])
print('Found', len(csv_files), 'monthly CSV files')
for f in csv_files:
    print(' *', os.path.basename(f))

In [ ]:
def load_data(csv_files, full=False, sample_n=500_000):
    """Load and concatenate all monthly CSVs."""
    dfs = []
    for fp in csv_files:
        if full:
            chunk = pd.read_csv(fp, low_memory=False)
        else:
            # Stratified sample across months
            per_file = sample_n // len(csv_files)
            chunk = pd.read_csv(fp, nrows=per_file, low_memory=False)
        dfs.append(chunk)
        print(f'  Loaded {os.path.basename(fp)}: {len(chunk):,} rows')
    df = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal rows: {len(df):,} | Columns: {df.shape[1]}')
    return df

df_raw = load_data(csv_files, full=FULL_DATA, sample_n=SAMPLE_N)

# Load zone lookup
zones = pd.read_csv(ZONE_FILE)
print(f'\nZone data: {zones.shape}')
zones.head()

## 2. Data Cleaning (Anomaly Handling)

We systematically handle each anomaly type and **quantify its impact** as required by the competition.

In [ ]:
df = df_raw.copy()
initial_rows = len(df)
print(f'Starting rows: {initial_rows:,}')
cleaning_log = {}

# ── Parse timestamps ──────────────────────────────────────
df['pickup_timestamp']  = pd.to_datetime(df['pickup_timestamp'],  errors='coerce')
df['dropoff_timestamp'] = pd.to_datetime(df['dropoff_timestamp'], errors='coerce')

# Derived: trip duration in minutes
df['trip_duration_min'] = (df['dropoff_timestamp'] - df['pickup_timestamp']).dt.total_seconds() / 60

# ── Anomaly 1: Drop-off before pickup (negative duration) ──
mask = df['trip_duration_min'] <= 0
pct = mask.sum() / initial_rows * 100
cleaning_log['Negative/Zero duration'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[1] Negative/zero duration removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 2: Negative fares ─────────────────────────────
mask = df['base_fare'] < 0
pct = mask.sum() / initial_rows * 100
cleaning_log['Negative base_fare'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[2] Negative base_fare removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 3: Distance = 0 with non-zero fare ────────────
mask = (df['distance_miles'] == 0) & (df['base_fare'] > 0)
pct = mask.sum() / initial_rows * 100
cleaning_log['Zero dist + nonzero fare'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[3] Zero distance + non-zero fare removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 4: Passenger count = 0 ───────────────────────
mask = df['rider_count'] <= 0
pct = mask.sum() / initial_rows * 100
cleaning_log['Passenger count <= 0'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[4] Passenger count <= 0 removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 5: Unrealistic speed (> 120 mph or == 0 for non-zero distance) ──
df['speed_mph'] = df['distance_miles'] / (df['trip_duration_min'] / 60)
mask = (df['speed_mph'] > 120) | (df['speed_mph'].isnull())
pct = mask.sum() / initial_rows * 100
cleaning_log['Unrealistic speed (>120 mph)'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[5] Unrealistic speed removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 6: Extreme trip durations (> 5 hours) ────────
mask = df['trip_duration_min'] > 300
pct = mask.sum() / initial_rows * 100
cleaning_log['Trip > 5 hours'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[6] Trip duration > 5h removed: {mask.sum():,} rows ({pct:.3f}%)')

# ── Anomaly 7: Voided / No-charge trips (not real fares) ──
mask = df['fare_settlement_method'].isin([3, 6])  # No charge, Voided
pct = mask.sum() / initial_rows * 100
cleaning_log['Voided/No-charge trips'] = {'count': mask.sum(), 'pct': round(pct, 3)}
df = df[~mask]
print(f'[7] Voided/No-charge trips removed: {mask.sum():,} rows ({pct:.3f}%)')

final_rows = len(df)
total_removed = initial_rows - final_rows
print(f'\n✅ Clean dataset: {final_rows:,} rows')
print(f'   Total removed: {total_removed:,} ({total_removed/initial_rows*100:.2f}%)')

In [ ]:
# Cleaning impact summary
log_df = pd.DataFrame(cleaning_log).T.reset_index()
log_df.columns = ['Anomaly Type', 'Rows Removed', '% of Total']
log_df = log_df.sort_values('Rows Removed', ascending=False)
print('\n📋 Cleaning Summary:')
display(log_df)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(log_df['Anomaly Type'], log_df['% of Total'], color='tomato')
ax.set_xlabel('% of Total Rows Removed')
ax.set_title('Anomaly Impact on Dataset')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'cleaning_impact.png'), dpi=150)
plt.show()

## 3. Feature Engineering

We extract temporal, spatial, and trip-level features that are **available before the trip starts** (for fair pre-trip prediction).

In [ ]:
# ── Temporal features (from pickup_timestamp only — pre-trip info) ──
df['pickup_hour']       = df['pickup_timestamp'].dt.hour
df['pickup_dayofweek']  = df['pickup_timestamp'].dt.dayofweek   # 0=Mon, 6=Sun
df['pickup_month']      = df['pickup_timestamp'].dt.month
df['pickup_dayofyear']  = df['pickup_timestamp'].dt.dayofyear
df['is_weekend']        = (df['pickup_dayofweek'] >= 5).astype(int)
df['is_rush_hour']      = df['pickup_hour'].isin([7,8,9,17,18,19]).astype(int)
df['is_night']          = df['pickup_hour'].isin(range(22, 24)).astype(int) | df['pickup_hour'].isin(range(0, 6)).astype(int)

# ── Spatial features ─────────────────────────────────────
# Join zone names for pickup and dropoff
df = df.merge(zones[['loc_id','borough_name','service_zone']].rename(
    columns={'loc_id':'origin_loc_id','borough_name':'pickup_borough','service_zone':'pickup_service_zone'}),
    on='origin_loc_id', how='left')

df = df.merge(zones[['loc_id','borough_name','service_zone']].rename(
    columns={'loc_id':'dest_loc_id','borough_name':'dropoff_borough','service_zone':'dropoff_service_zone'}),
    on='dest_loc_id', how='left')

# Airport flag (JFK=132, LGA=138, EWR=1)
AIRPORT_ZONES = {132, 138, 1}
df['is_airport_pickup']  = df['origin_loc_id'].isin(AIRPORT_ZONES).astype(int)
df['is_airport_dropoff'] = df['dest_loc_id'].isin(AIRPORT_ZONES).astype(int)

# Inter-borough trip
df['inter_borough'] = (df['pickup_borough'] != df['dropoff_borough']).astype(int)

# ── Encode categoricals ───────────────────────────────────
le = LabelEncoder()
for col in ['pickup_borough','dropoff_borough','pickup_service_zone','dropoff_service_zone','offline_record_flag']:
    df[col] = df[col].fillna('Unknown')
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

print('Features created:', df.shape[1])
print('\nSample feature preview:')
df[['pickup_hour','is_weekend','is_rush_hour','pickup_borough','dropoff_borough',
    'inter_borough','is_airport_pickup','trip_duration_min']].head(3)

## 4. Task 2.1 — Base Fare Prediction

> **Business Goal:** "No-Surprises Upfront Pricing" — predict `base_fare` before the trip starts.
> 
> **Inputs available pre-trip:** origin zone, destination zone, distance estimate, rider count, time of day, day of week, rate class.

In [ ]:
# ── Feature set for fare prediction (PRE-TRIP only) ──────
FARE_FEATURES = [
    'distance_miles',
    'rider_count',
    'rate_class_id',
    'origin_loc_id',
    'dest_loc_id',
    'pickup_hour',
    'pickup_dayofweek',
    'pickup_month',
    'is_weekend',
    'is_rush_hour',
    'is_night',
    'is_airport_pickup',
    'is_airport_dropoff',
    'inter_borough',
    'pickup_borough_enc',
    'dropoff_borough_enc',
    'pickup_service_zone_enc',
    'dropoff_service_zone_enc',
    'toll_total',            # known for route (toll roads)
]

TARGET_FARE = 'base_fare'

df_fare = df[FARE_FEATURES + [TARGET_FARE]].dropna()
print(f'Fare modeling dataset: {df_fare.shape}')
print(f'Target distribution:')
print(df_fare[TARGET_FARE].describe())

In [ ]:
# ── Train / Val / Test split (70 / 15 / 15) ──────────────
X_fare = df_fare[FARE_FEATURES]
y_fare = df_fare[TARGET_FARE]

X_temp, X_fare_test, y_temp, y_fare_test = train_test_split(
    X_fare, y_fare, test_size=0.15, random_state=SEED)
X_fare_train, X_fare_val, y_fare_train, y_fare_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, random_state=SEED)

print(f'Train: {X_fare_train.shape[0]:,} | Val: {X_fare_val.shape[0]:,} | Test: {X_fare_test.shape[0]:,}')

In [ ]:
# ── Evaluation helper ─────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """Fit, evaluate on val + test, return metrics dict."""
    model.fit(X_tr, y_tr)
    results = {}
    for split_name, X_s, y_s in [('Val', X_val, y_val), ('Test', X_te, y_te)]:
        preds = model.predict(X_s)
        rmse  = np.sqrt(mean_squared_error(y_s, preds))
        mae   = mean_absolute_error(y_s, preds)
        r2    = r2_score(y_s, preds)
        results[split_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        print(f'  [{split_name}] {name} → RMSE: {rmse:.3f} | MAE: {mae:.3f} | R²: {r2:.4f}')
    return model, results

fare_results = {}

In [ ]:
# ── Model 1: Linear Regression (baseline) ─────────────────
print('--- Baseline: Linear Regression ---')
lr_fare, lr_fare_res = evaluate_model(
    'LinearRegression',
    Pipeline([('imputer', SimpleImputer(strategy='median')),
              ('scaler', StandardScaler()),
              ('model', LinearRegression())]),
    X_fare_train, y_fare_train, X_fare_val, y_fare_val, X_fare_test, y_fare_test
)
fare_results['LinearRegression'] = lr_fare_res

In [ ]:
# ── Model 2: XGBoost ──────────────────────────────────────
print('--- XGBoost ---')
xgb_fare_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    tree_method='hist',
    n_jobs=-1
)
xgb_fare, xgb_fare_res = evaluate_model(
    'XGBoost',
    xgb_fare_model,
    X_fare_train, y_fare_train, X_fare_val, y_fare_val, X_fare_test, y_fare_test
)
fare_results['XGBoost'] = xgb_fare_res

In [ ]:
# ── Model 3: LightGBM ─────────────────────────────────────
print('--- LightGBM ---')
lgb_fare_model = lgb.LGBMRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgb_fare, lgb_fare_res = evaluate_model(
    'LightGBM',
    lgb_fare_model,
    X_fare_train, y_fare_train, X_fare_val, y_fare_val, X_fare_test, y_fare_test
)
fare_results['LightGBM'] = lgb_fare_res

In [ ]:
# ── Model Comparison Table ────────────────────────────────
rows = []
for name, res in fare_results.items():
    for split, metrics in res.items():
        rows.append({'Model': name, 'Split': split, **metrics})
fare_compare = pd.DataFrame(rows)
print('\n📊 Fare Prediction — Model Comparison:')
display(fare_compare)

# Pick best model by Test RMSE
best_fare_name = fare_compare[fare_compare['Split']=='Test'].sort_values('RMSE').iloc[0]['Model']
print(f'\n🏆 Best fare model: {best_fare_name}')

In [ ]:
# ── Best model: Feature Importance ───────────────────────
best_fare_models = {'LinearRegression': lr_fare, 'XGBoost': xgb_fare, 'LightGBM': lgb_fare}
best_fare_model  = best_fare_models[best_fare_name]

# Feature importance (XGB / LGB have .feature_importances_)
if hasattr(best_fare_model, 'feature_importances_'):
    fi = pd.Series(best_fare_model.feature_importances_, index=FARE_FEATURES)
    fi = fi.sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    fi.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{best_fare_name} — Top Feature Importances (Fare Prediction)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'fare_feature_importance.png'), dpi=150)
    plt.show()

In [ ]:
# ── Predicted vs Actual scatter ───────────────────────────
y_pred_fare = best_fare_model.predict(X_fare_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_fare_test[:2000], y_pred_fare[:2000], alpha=0.3, s=8, color='steelblue')
axes[0].plot([0, y_fare_test.max()], [0, y_fare_test.max()], 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual base_fare ($)')
axes[0].set_ylabel('Predicted base_fare ($)')
axes[0].set_title(f'{best_fare_name}: Predicted vs Actual')
axes[0].legend()

# Residuals
residuals = y_fare_test.values - y_pred_fare
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', lw=2)
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'fare_pred_vs_actual.png'), dpi=150)
plt.show()

### 📝 Metric Justification — Fare Prediction

| Metric | Why we use it |
|---|---|
| **RMSE** | Penalises large fare errors more heavily — crucial because a large surprise fare harms passenger trust more than a small one |
| **MAE** | Easy to interpret in dollar terms — "on average our fare estimate is off by \$X" |
| **R²** | Shows what % of fare variance our model explains — gives a dimensionless quality score for comparing models |

For a real-time pricing engine, **RMSE is the primary metric** because passengers care most about avoiding big surprises.

## 5. Task 2.2 — Trip Duration Prediction

> **Business Goal:** "On-Time Arrival Estimator" — predict trip duration in minutes before the trip starts.

In [ ]:
# ── Feature set for duration prediction (PRE-TRIP only) ──
DUR_FEATURES = [
    'distance_miles',
    'rider_count',
    'rate_class_id',
    'origin_loc_id',
    'dest_loc_id',
    'pickup_hour',
    'pickup_dayofweek',
    'pickup_month',
    'is_weekend',
    'is_rush_hour',
    'is_night',
    'is_airport_pickup',
    'is_airport_dropoff',
    'inter_borough',
    'pickup_borough_enc',
    'dropoff_borough_enc',
    'pickup_service_zone_enc',
    'dropoff_service_zone_enc',
    'toll_total',
]

TARGET_DUR = 'trip_duration_min'

df_dur = df[DUR_FEATURES + [TARGET_DUR]].dropna()
print(f'Duration modeling dataset: {df_dur.shape}')
print(f'Target distribution (minutes):')
print(df_dur[TARGET_DUR].describe())

In [ ]:
# ── Train / Val / Test split ──────────────────────────────
X_dur = df_dur[DUR_FEATURES]
y_dur = df_dur[TARGET_DUR]

X_temp2, X_dur_test, y_temp2, y_dur_test = train_test_split(
    X_dur, y_dur, test_size=0.15, random_state=SEED)
X_dur_train, X_dur_val, y_dur_train, y_dur_val = train_test_split(
    X_temp2, y_temp2, test_size=0.15/0.85, random_state=SEED)

print(f'Train: {X_dur_train.shape[0]:,} | Val: {X_dur_val.shape[0]:,} | Test: {X_dur_test.shape[0]:,}')

dur_results = {}

In [ ]:
# ── Model 1: Linear Regression (baseline) ─────────────────
print('--- Baseline: Linear Regression ---')
lr_dur, lr_dur_res = evaluate_model(
    'LinearRegression',
    Pipeline([('imputer', SimpleImputer(strategy='median')),
              ('scaler', StandardScaler()),
              ('model', LinearRegression())]),
    X_dur_train, y_dur_train, X_dur_val, y_dur_val, X_dur_test, y_dur_test
)
dur_results['LinearRegression'] = lr_dur_res

In [ ]:
# ── Model 2: XGBoost ──────────────────────────────────────
print('--- XGBoost ---')
xgb_dur_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    tree_method='hist',
    n_jobs=-1
)
xgb_dur, xgb_dur_res = evaluate_model(
    'XGBoost',
    xgb_dur_model,
    X_dur_train, y_dur_train, X_dur_val, y_dur_val, X_dur_test, y_dur_test
)
dur_results['XGBoost'] = xgb_dur_res

In [ ]:
# ── Model 3: LightGBM ─────────────────────────────────────
print('--- LightGBM ---')
lgb_dur_model = lgb.LGBMRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgb_dur, lgb_dur_res = evaluate_model(
    'LightGBM',
    lgb_dur_model,
    X_dur_train, y_dur_train, X_dur_val, y_dur_val, X_dur_test, y_dur_test
)
dur_results['LightGBM'] = lgb_dur_res

In [ ]:
# ── Model Comparison Table ────────────────────────────────
rows2 = []
for name, res in dur_results.items():
    for split, metrics in res.items():
        rows2.append({'Model': name, 'Split': split, **metrics})
dur_compare = pd.DataFrame(rows2)
print('\n📊 Duration Prediction — Model Comparison:')
display(dur_compare)

best_dur_name = dur_compare[dur_compare['Split']=='Test'].sort_values('RMSE').iloc[0]['Model']
print(f'\n🏆 Best duration model: {best_dur_name}')

In [ ]:
# ── Feature Importance ────────────────────────────────────
best_dur_models = {'LinearRegression': lr_dur, 'XGBoost': xgb_dur, 'LightGBM': lgb_dur}
best_dur_model  = best_dur_models[best_dur_name]

if hasattr(best_dur_model, 'feature_importances_'):
    fi2 = pd.Series(best_dur_model.feature_importances_, index=DUR_FEATURES)
    fi2 = fi2.sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    fi2.plot(kind='barh', ax=ax, color='mediumseagreen')
    ax.set_title(f'{best_dur_name} — Top Feature Importances (Duration Prediction)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'dur_feature_importance.png'), dpi=150)
    plt.show()

In [ ]:
# ── Predicted vs Actual ───────────────────────────────────
y_pred_dur = best_dur_model.predict(X_dur_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_dur_test[:2000], y_pred_dur[:2000], alpha=0.3, s=8, color='mediumseagreen')
axes[0].plot([0, y_dur_test.max()], [0, y_dur_test.max()], 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual duration (min)')
axes[0].set_ylabel('Predicted duration (min)')
axes[0].set_title(f'{best_dur_name}: Predicted vs Actual')
axes[0].legend()

residuals2 = y_dur_test.values - y_pred_dur
axes[1].hist(residuals2, bins=60, color='gold', edgecolor='white')
axes[1].axvline(0, color='black', lw=2)
axes[1].set_xlabel('Residual (Actual − Predicted) minutes')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dur_pred_vs_actual.png'), dpi=150)
plt.show()

### 📝 Metric Justification — Trip Duration Prediction

| Metric | Why we use it |
|---|---|
| **RMSE** | Penalises large ETA errors — a prediction off by 20 min is far worse than one off by 2 min |
| **MAE** | Tells passengers the typical error in plain minutes: "your ETA is accurate within ±X minutes" |
| **R²** | Captures how much of the duration variability our model explains vs. a naive mean baseline |

## 6. Save Dataset Splits & Export Models

In [ ]:
# ── Save train / val / test splits ───────────────────────
SPLITS_DIR = os.path.join(MODEL_DIR, 'splits')
os.makedirs(SPLITS_DIR, exist_ok=True)

# Fare splits
pd.concat([X_fare_train, y_fare_train], axis=1).to_csv(os.path.join(SPLITS_DIR, 'fare_train.csv'), index=False)
pd.concat([X_fare_val,   y_fare_val],   axis=1).to_csv(os.path.join(SPLITS_DIR, 'fare_val.csv'),   index=False)
pd.concat([X_fare_test,  y_fare_test],  axis=1).to_csv(os.path.join(SPLITS_DIR, 'fare_test.csv'),  index=False)

# Duration splits
pd.concat([X_dur_train, y_dur_train], axis=1).to_csv(os.path.join(SPLITS_DIR, 'duration_train.csv'), index=False)
pd.concat([X_dur_val,   y_dur_val],   axis=1).to_csv(os.path.join(SPLITS_DIR, 'duration_val.csv'),   index=False)
pd.concat([X_dur_test,  y_dur_test],  axis=1).to_csv(os.path.join(SPLITS_DIR, 'duration_test.csv'),  index=False)

print('✅ Dataset splits saved to:', SPLITS_DIR)

In [ ]:
# ── Export trained models as .pkl ────────────────────────
fare_pkl_path = os.path.join(MODEL_DIR, f'fare_model_{best_fare_name.lower()}.pkl')
dur_pkl_path  = os.path.join(MODEL_DIR, f'duration_model_{best_dur_name.lower()}.pkl')

joblib.dump(best_fare_model, fare_pkl_path)
joblib.dump(best_dur_model,  dur_pkl_path)

print(f'✅ Fare model saved:     {fare_pkl_path}')
print(f'✅ Duration model saved: {dur_pkl_path}')

In [ ]:
# ── Save feature lists alongside model (for inference) ───
import json
meta = {
    'fare_features':     FARE_FEATURES,
    'duration_features': DUR_FEATURES,
    'fare_target':       TARGET_FARE,
    'duration_target':   TARGET_DUR,
    'best_fare_model':   best_fare_name,
    'best_dur_model':    best_dur_name,
}
with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Model metadata saved')

## 7. Final Summary

In [ ]:
print('=' * 55)
print('  MEMBER 2 — PREDICTIVE MODELING SUMMARY')
print('=' * 55)

print('\n📌 Task 2.1: Base Fare Prediction')
fare_test = fare_compare[fare_compare['Split'] == 'Test']
best_fare_row = fare_test[fare_test['Model'] == best_fare_name].iloc[0]
print(f'   Best Model : {best_fare_name}')
print(f'   RMSE       : ${best_fare_row["RMSE"]:.3f}')
print(f'   MAE        : ${best_fare_row["MAE"]:.3f}')
print(f'   R²         : {best_fare_row["R2"]:.4f}')

print('\n📌 Task 2.2: Trip Duration Prediction')
dur_test = dur_compare[dur_compare['Split'] == 'Test']
best_dur_row = dur_test[dur_test['Model'] == best_dur_name].iloc[0]
print(f'   Best Model : {best_dur_name}')
print(f'   RMSE       : {best_dur_row["RMSE"]:.3f} min')
print(f'   MAE        : {best_dur_row["MAE"]:.3f} min')
print(f'   R²         : {best_dur_row["R2"]:.4f}')

print('\n📦 Saved Artifacts:')
for f in os.listdir(MODEL_DIR):
    full = os.path.join(MODEL_DIR, f)
    if os.path.isfile(full):
        size = os.path.getsize(full) / 1024
        print(f'   {f} ({size:.1f} KB)')

print('\n✅ Member 2 tasks complete!')